In [ ]:
# Cell	Purpose
# 1	Install dependencies
# 2	Load model
# 3	Test generation
# 4	Load SAE
# 5	Extract activations
# 6	Top features for last token
# 7	Steering demo
# 8	Compare features across prompts
# 9	Feature identification
# 10	Find Golden Gate Bridge feature
# 11	Steer with Golden Gate Bridge feature

In [1]:
# [Cell 1] Install dependencies
!pip install transformer_lens sae_lens transformers torch
# !pip install --force-reinstall numpy scipy pandas scikit-learn

# ⚠️ If on Colab, restart runtime after install:
# Runtime → Restart runtime (Ctrl+M+.), then run from Cell 2

  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached sentencepiece-0.2.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (10 kB)
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of transformers to determine which version is compatible with other requirements. This could take a while.
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.7/739.7 kB 1.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 1.7 MB/s  0:00:0036m-:--:--
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 1.9 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 1.2 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.3/25.3 MB 1.6 MB/s  0:00:15m0:00:0100:01m
Using cached sentenc

In [1]:
# [Cell 2] Load model (local CPU)
import torch
import transformer_lens as tl
import os
# from google.colab import drive

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using: {device}')

# --- Colab (Google Drive) ---
# drive.mount('/content/drive')
# save_path = '/content/drive/MyDrive/gpt2_small_hooked.pt'

# --- Local ---
save_dir = os.path.dirname(os.path.abspath('__file__'))
save_path = os.path.join(save_dir, 'gpt2_small_hooked.pt')

if os.path.exists(save_path):
    model = tl.HookedTransformer.from_pretrained('gpt2-small', device='cpu')
    model.load_state_dict(torch.load(save_path, map_location=device))
    model.to(device)
    print("Loaded from local cache")
else:
    model = tl.HookedTransformer.from_pretrained('gpt2-small', device=device)
    torch.save(model.state_dict(), save_path)
    print(f"Downloaded and saved to {save_path}")

print(f'GPT-2 small: {sum(p.numel() for p in model.parameters())/1e6:.0f}M params')
print(f'dim={model.cfg.d_model}, layers={model.cfg.n_layers}, heads={model.cfg.n_heads}')

Using: cpu


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded pretrained model gpt2-small into HookedTransformer
Downloaded and saved to /home/cody/Documents/Anthropic/mechanistic_interpretability/gpt2/gpt2_small_hooked.pt
GPT-2 small: 163M params
dim=768, layers=12, heads=12


In [ ]:
# [Cell 3] Test generation
prompt = "Hi, Who are you?"
output = model.generate(prompt, max_new_tokens=100, temperature=0.8)
print(output)

  0%|          | 0/100 [00:00<?, ?it/s]

Hi, Who are you? I am Eric. I am the CEO and Publisher of Starbreeze Games. I am currently working on the full engine. I have written three books about Starbreeze and this is the first time I have spent time here. I love games, I love viral reviews and I love making great games. I watch directed videos and I LOVE to write my own games. So here I am two weeks away from finishing the Starbreeze release. I'm excited to finally be working on


## SAE Feature Extraction
Load a pretrained Sparse Autoencoder from SAE Lens and extract features from GPT-2's residual stream.

In [5]:
# [Cell 4] Load SAE
from sae_lens import SAE

sae, cfg_dict, sparsity = SAE.from_pretrained(
    release="gpt2-small-res-jb",       # Joseph Bloom's GPT-2 SAEs
    sae_id="blocks.8.hook_resid_pre",   # layer 8 residual stream
    device=device
)

print(f"SAE: {sae.cfg.d_in}d input → {sae.cfg.d_sae}d features")
print(f"Expansion factor: {sae.cfg.d_sae // sae.cfg.d_in}x")

cfg.json: 0.00B [00:00, ?B/s]

blocks.8.hook_resid_pre/sae_weights.safe(…):   0%|          | 0.00/151M [00:00<?, ?B/s]

blocks.8.hook_resid_pre/sparsity.safeten(…):   0%|          | 0.00/98.4k [00:00<?, ?B/s]

SAE: 768d input → 24576d features
Expansion factor: 32x


/home/cody/anaconda3/lib/python3.12/site-packages/sae_lens/saes/sae.py:251: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(
/home/cody/.cache/pip_tmp/ipykernel_27783/1924163614.py:4: DeprecationWarning: Unpacking SAE objects is deprecated. SAE.from_pretrained() now returns only the SAE object. Use SAE.from_pretrained_with_cfg_and_sparsity() to get the config dict and sparsity as well.
  sae, cfg_dict, sparsity = SAE.from_pretrained(


In [7]:
# [Cell 5] Extract activations + SAE features
test_prompt = "The capital of France is"
_, cache = model.run_with_cache(test_prompt)

# Get residual stream activations at layer 8
resid = cache["blocks.8.hook_resid_pre"]  # shape: [batch, seq_len, d_model]
print(f"Residual stream shape: {resid.shape}")

# Encode through SAE → sparse feature activations
feature_acts = sae.encode(resid)  # shape: [batch, seq_len, d_sae]
print(f"Feature activations shape: {feature_acts.shape}")

# How many features fire per token?
tokens = model.to_str_tokens(test_prompt)
for i, tok in enumerate(tokens):
    n_active = (feature_acts[0, i] > 0).sum().item()
    print(f"  Token '{tok}': {n_active} active features out of {sae.cfg.d_sae}")

Residual stream shape: torch.Size([1, 6, 768])
Feature activations shape: torch.Size([1, 6, 24576])
  Token '<|endoftext|>': 8 active features out of 24576
  Token 'The': 8 active features out of 24576
  Token ' capital': 35 active features out of 24576
  Token ' of': 66 active features out of 24576
  Token ' France': 47 active features out of 24576
  Token ' is': 51 active features out of 24576


In [8]:
# [Cell 6] Top features for last token
last_token_acts = feature_acts[0, -1]  # features for " is"
top_k = 10
top_features = torch.topk(last_token_acts, top_k)

print(f"Top {top_k} features firing on last token '{tokens[-1]}':")
print(f"{'Feature ID':>12} | {'Activation':>10}")
print("-" * 27)
for feat_id, act_val in zip(top_features.indices, top_features.values):
    print(f"{feat_id.item():>12} | {act_val.item():>10.3f}")

Top 10 features firing on last token ' is':
  Feature ID | Activation
---------------------------
       21000 |     10.298
       18220 |      8.410
       14430 |      7.472
        1442 |      6.797
       19805 |      6.314
       21496 |      6.313
        6863 |      6.239
       12013 |      5.085
       19182 |      4.214
       15547 |      3.901


## Feature Steering
Inject a specific SAE feature into the residual stream during generation to steer the model's output.

In [10]:
# [Cell 7] Steering demo
steering_feature_id = top_features.indices[0].item()
steering_vector = sae.W_dec[steering_feature_id]  # decoder direction for this feature
coeff = 10.0  # steering strength (try 5-50)

def steering_hook(resid, hook):
    resid[:, :, :] += coeff * steering_vector
    return resid

# Generate WITHOUT steering
prompt = "I think the best city in the world is"
print("=== Without steering ===")
output_normal = model.generate(prompt, max_new_tokens=50, temperature=0.7)
print(output_normal)

# Generate WITH steering
print(f"\n=== With steering (feature {steering_feature_id}, coeff={coeff}) ===")
with model.hooks(fwd_hooks=[("blocks.8.hook_resid_pre", steering_hook)]):
    output_steered = model.generate(prompt, max_new_tokens=50, temperature=0.7)
print(output_steered)

=== Without steering ===


  0%|          | 0/50 [00:00<?, ?it/s]

I think the best city in the world is Tokyo.

It's called the Tokyo Metro. It is the Japanese version of the Tokyo subway system, although it's much more efficient.

Tens of thousands of people walk in this city every day. It's the most efficient way

=== With steering (feature 21000, coeff=10.0) ===


  0%|          | 0/50 [00:00<?, ?it/s]

I think the best city in the world is the city of New York.

I don't know if it's the New York of the world or the New York of the soul.

Brooklyn is almost like a city with a lot of restaurants, a lot of restaurants, and


In [11]:
# [Cell 8] Compare features across prompts
prompts = [
    "The president of the United States",
    "The Eiffel Tower in Paris",
    "Machine learning algorithms can",
    "The recipe for chocolate cake",
]

for p in prompts:
    _, c = model.run_with_cache(p)
    acts = sae.encode(c["blocks.8.hook_resid_pre"])
    last_acts = acts[0, -1]
    top3 = torch.topk(last_acts, 3)
    toks = model.to_str_tokens(p)
    print(f"'{toks[-1]}' ← \"{p}\"")
    for fid, fval in zip(top3.indices, top3.values):
        print(f"    feature {fid.item():>6}: {fval.item():.3f}")
    print()

' States' ← "The president of the United States"
    feature  17228: 39.430
    feature    939: 14.309
    feature  14697: 8.916

' Paris' ← "The Eiffel Tower in Paris"
    feature    974: 44.852
    feature  18593: 12.430
    feature   7329: 7.324

' can' ← "Machine learning algorithms can"
    feature  18695: 30.987
    feature  20933: 13.372
    feature   6072: 10.026

' cake' ← "The recipe for chocolate cake"
    feature  22176: 40.843
    feature   9976: 10.023
    feature   7161: 7.233



## Feature Identification
Run diverse prompts through the model and find which ones maximally activate a given feature — this tells you what concept that feature represents.

In [12]:
# [Cell 9] Feature identification — what does a feature represent?
FEATURE_ID = 974  # ← Change this to any feature ID you want to investigate

probe_prompts = [
    "The Eiffel Tower in Paris is a famous landmark",
    "London is the capital of England",
    "The president of the United States lives in Washington",
    "Tokyo is a massive city in Japan",
    "Berlin is the capital of Germany",
    "I love eating pizza and pasta in Rome",
    "Sydney Opera House is in Australia",
    "The Great Wall of China is ancient",
    "Moscow is the capital of Russia",
    "Cairo has the pyramids of Egypt",
    "Machine learning models are trained on data",
    "The stock market crashed in 2008",
    "Photosynthesis converts sunlight into energy",
    "Shakespeare wrote plays in the 16th century",
    "The recipe calls for butter and sugar",
    "DNA stores genetic information in cells",
    "The French Revolution started in 1789",
    "Soccer is the most popular sport worldwide",
    "Quantum mechanics describes subatomic particles",
    "The Amazon rainforest produces oxygen",
    "Marie Curie discovered radium in Paris",
    "Bitcoin is a decentralized cryptocurrency",
    "The speed of light is constant",
    "Notre Dame cathedral is in Paris France",
    "Croissants and baguettes are French pastries",
]

results = []
for p in probe_prompts:
    toks = model.to_str_tokens(p)
    _, c = model.run_with_cache(p)
    acts = sae.encode(c["blocks.8.hook_resid_pre"])
    for i, tok in enumerate(toks):
        val = acts[0, i, FEATURE_ID].item()
        if val > 0:
            results.append((val, tok, p))

results.sort(reverse=True)

print(f"Feature {FEATURE_ID} — top activating tokens:\n")
print(f"{'Activation':>10} | {'Token':>12} | Prompt")
print("-" * 70)
for val, tok, prompt in results[:15]:
    print(f"{val:>10.3f} | {tok:>12} | {prompt}")

Feature 974 — top activating tokens:

Activation |        Token | Prompt
----------------------------------------------------------------------
    44.852 |        Paris | The Eiffel Tower in Paris is a famous landmark
    43.933 |        Paris | Notre Dame cathedral is in Paris France
    42.795 |        Paris | Marie Curie discovered radium in Paris
     4.070 |          lin | Berlin is the capital of Germany
     1.234 |       France | Notre Dame cathedral is in Paris France
     1.004 |          Tok | Tokyo is a massive city in Japan


In [19]:
# [Cell 12] Search for features by concept
# Type a concept → see which feature IDs control it

CONCEPT = "Golden Gate Bridge"  # ← Change this to any word or concept

# Generate prompts containing the concept
search_prompts = [
    f"The {CONCEPT} is over water cool",
    f"The {CONCEPT} is something that everyone should visit",
    f"The {CONCEPT} overlooks the bay",
    f" People often associate {CONCEPT} with",
]

# Find which features fire on/around the concept word(s)
from collections import Counter
feature_counts = Counter()
feature_max_act = {}

concept_words = [w.lower() for w in CONCEPT.split()]

for p in search_prompts:
    toks = model.to_str_tokens(p)
    _, c = model.run_with_cache(p)
    acts = sae.encode(c["blocks.8.hook_resid_pre"])
    for i, tok in enumerate(toks):
        if any(w in tok.strip().lower() for w in concept_words):
            top = torch.topk(acts[0, i], 10)
            for fid, fval in zip(top.indices, top.values):
                fid_int = fid.item()
                feature_counts[fid_int] += 1
                feature_max_act[fid_int] = max(feature_max_act.get(fid_int, 0), fval.item())

# Rank by frequency (features that appear across multiple prompts = more reliable)
print(f"Features for concept: \"{CONCEPT}\"\n")
print(f"{'Feature ID':>10} | {'Times seen':>10} | {'Max activation':>14}")
print("-" * 42)
for fid, count in feature_counts.most_common(15):
    print(f"{fid:>10} | {count:>10} | {feature_max_act[fid]:>14.3f}")

Features for concept: "Golden Gate Bridge"

Feature ID | Times seen | Max activation
------------------------------------------
      7525 |          8 |          8.126
       735 |          7 |         12.790
     11744 |          6 |         10.205
      7451 |          6 |          4.590
     10261 |          4 |         73.706
      4587 |          4 |         10.941
      8198 |          4 |         24.337
     17840 |          4 |         16.055
     23937 |          4 |         10.385
      3808 |          4 |          6.922
       990 |          4 |          6.821
     21665 |          4 |          5.795
      9752 |          4 |          5.553
     17608 |          4 |         39.803
      5055 |          4 |          6.894


In [24]:
# [Cell 10] Steer with Golden Gate Bridge features
# Combine multiple related features for stronger topical steering
gg_features = {
    # 10261: 15.0,  # "Gate"
    # 17608: 15.0,  # "Bridge"
    # 20677: 15.0,  # "Francisco"
    # 9752:  15.0,  # "Bay"
    7525:  50.0,  # "Bay Area"
}

# Build combined steering vector
steering_vector = sum(
    coeff * sae.W_dec[fid] for fid, coeff in gg_features.items()
)

def gg_steering_hook(resid, hook):
    resid[:, :, :] += steering_vector
    return resid

prompt = "I think the best place to visit is"
print("======= Without steering =======")
print(model.generate(prompt, max_new_tokens=50, temperature=0.7))

print(f"\n======= With steering =======")
with model.hooks(fwd_hooks=[("blocks.8.hook_resid_pre", gg_steering_hook)]):
    print(model.generate(prompt, max_new_tokens=50, temperature=0.7))

======= Without steering =======


  0%|          | 0/50 [00:00<?, ?it/s]

I think the best place to visit is a bookshop in London. It's a bit cheaper than a bookshop in the US, or in Portugal. If I wanted to buy a book, I would have to go to a bookstore in Amsterdam, because if I tried to buy around the

======= With steering =======


  0%|          | 0/50 [00:00<?, ?it/s]

I think the best place to visit is the 3-5, mostly-2-5, city-based, for-sale shop's best place to work has just shifted East-native's vibe to a more hip-pop, nod hip music scene.

The thrum-
